# Marketing Campaign Analysis — Data Cleaning & Feature Engineering

**Notebook 02 di 03** · Pulizia del dataset e costruzione delle feature analitiche

## Obiettivo di questo notebook
Trasformare il dataset grezzo esplorato nel Notebook 01 in un dataset pronto per l'analisi, applicando le correzioni identificate e arricchendolo con feature derivate utili alla segmentazione clienti e all'analisi dell'efficacia delle campagne.

## Operazioni previste
1. **Rimozione colonne costanti** (`Z_CostContact`, `Z_Revenue`) — prive di valore analitico
2. **Rimozione outlier strutturali** — 3 record con `Year_Birth` < 1940 e il singolo record con `Income` = 666.666
3. **Imputazione missing values** in `Income` (24 record) con mediana per gruppo `Education`
4. **Conversione `Dt_Customer`** in formato `datetime`
5. **Consolidamento `Marital_Status`** in due categorie con significato di business (`In_Couple` / `Single`)
6. **Feature engineering**: creazione di 7 variabili derivate (`Age`, `MntTotal`, `TotalPurchases`, `TotalCampaignsAccepted`, `Customer_Tenure_Days`, `Family_Size`, `Has_Children`)
7. **Salvataggio** del dataset pulito in `data/processed/marketing_campaign_clean.csv`

## Criterio di ordine
L'ordine delle operazioni non è casuale: rimuoviamo *prima* outlier e colonne costanti, *poi* imputiamo i missing. Questo evita che la mediana usata per l'imputazione venga distorta dall'outlier `Income` = 666.666, che altrimenti gonfierebbe leggermente la mediana del gruppo `Graduation`.

---

## 0. Setup: librerie e caricamento del dataset grezzo

Ripartiamo dal CSV originale. Non usiamo il DataFrame del Notebook 01 perché ogni notebook deve essere **eseguibile indipendentemente**: è una buona pratica che garantisce riproducibilità e semplifica il debug.

In [1]:
# Librerie di data manipulation
import pandas as pd
import numpy as np

# Configurazioni di visualizzazione output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Path al file CSV grezzo (stesso del Notebook 01)
DATA_PATH = '../data/marketing_campaign.csv'

# Caricamento (separatore tab, non virgola)
df = pd.read_csv(DATA_PATH, sep='\t')

print(f"Dataset grezzo caricato: {df.shape[0]:,} righe x {df.shape[1]} colonne")

Dataset grezzo caricato: 2,240 righe x 29 colonne


## 1. Rimozione delle colonne costanti

Nel Notebook 01 abbiamo identificato due colonne senza variabilità: `Z_CostContact` (sempre 3) e `Z_Revenue` (sempre 11). Sono probabili metadati residui del business case iFood originale e non portano informazione utile all'analisi. Le rimuoviamo.

In [2]:
# Rimozione delle due colonne costanti
cols_to_drop = ['Z_CostContact', 'Z_Revenue']
df = df.drop(columns=cols_to_drop)

print(f"Colonne rimosse: {cols_to_drop}")
print(f"Dataset ora: {df.shape[0]:,} righe x {df.shape[1]} colonne")

Colonne rimosse: ['Z_CostContact', 'Z_Revenue']
Dataset ora: 2,240 righe x 27 colonne


## 2. Rimozione outlier anagrafici

I 3 record con `Year_Birth` < 1940 implicano clienti di 85+ anni che, accoppiati con i dati di spesa attiva nel periodo 2012-2014, sono molto probabilmente errori di data entry o record di test. Li rimuoviamo applicando la soglia definita nel Notebook 01.

La soglia di 1940 è conservativa: taglia solo i 3 valori palesemente anomali (1893, 1899, 1900) senza toccare clienti anziani ma plausibili. Un cliente nato nel 1941 avrebbe ~73 anni nel 2014, un'età ancora compatibile con l'attività di acquisto osservata.

In [3]:
# Rimozione dei record con Year_Birth anomalo
mask_age_outliers = df['Year_Birth'] < 1940
n_removed_age = mask_age_outliers.sum()

print(f"Record rimossi per Year_Birth < 1940: {n_removed_age}")
print(f"Year_Birth eliminati: {sorted(df.loc[mask_age_outliers, 'Year_Birth'].tolist())}")

df = df.loc[~mask_age_outliers].copy()
print(f"\nDataset ora: {df.shape[0]:,} righe x {df.shape[1]} colonne")

Record rimossi per Year_Birth < 1940: 3
Year_Birth eliminati: [1893, 1899, 1900]

Dataset ora: 2,237 righe x 27 colonne


## 3. Rimozione dell'outlier di reddito

Il record con `Income` = 666.666 è un singoletto isolato: il secondo reddito più alto nel dataset è di 162.397, oltre **quattro volte inferiore**. La cifra "666666" ha tutta l'aria di essere un valore di test o un errore di inserimento (cifra ripetuta, numerologicamente sospetta). Lo rimuoviamo.

### Perché NON usiamo una regola IQR o percentile

Un approccio "statistico" (es. rimuovere tutti i record sopra il 99° percentile o oltre Q3 + 1.5·IQR) sarebbe apparentemente più rigoroso, ma in realtà controproducente per questo progetto: taglierebbe via il segmento di clienti premium (redditi 120-162k, tutti plausibili per profili PhD/Master), che è **proprio il segmento che vogliamo studiare** per l'allocazione del budget marketing. La rimozione chirurgica del singolo outlier patologico preserva il segnale high-value.

In [4]:
# Rimozione del singolo outlier di reddito
mask_income_outlier = df['Income'] == 666666
n_removed_income = mask_income_outlier.sum()

print(f"Record rimossi per Income = 666.666: {n_removed_income}")

df = df.loc[~mask_income_outlier].copy()
print(f"\nDataset ora: {df.shape[0]:,} righe x {df.shape[1]} colonne")
print(f"Nuovo massimo di Income: {df['Income'].max():,.0f} EUR")

Record rimossi per Income = 666.666: 1

Dataset ora: 2,236 righe x 27 colonne
Nuovo massimo di Income: 162,397 EUR


## 4. Imputazione dei missing in `Income`

I 24 record con `Income` mancante (1.07% del dataset) verranno imputati con la **mediana per gruppo `Education`**. La scelta è motivata da tre ragioni:

1. **Correlazione forte**: il livello di istruzione è uno dei predittori più robusti del reddito nei dataset socio-demografici. Imputare con la mediana del gruppo di appartenenza produce stime più accurate della mediana globale.
2. **Robustezza agli outlier**: usiamo la mediana (non la media) perché la distribuzione dei redditi è tipicamente asimmetrica a destra.
3. **Dimensione dei gruppi**: `Education` ha solo 5 categorie, ognuna con abbondanza di osservazioni (centinaia di record ciascuna). Nessun rischio di imputare basandoci su campioni troppo piccoli.

Svolgiamo la procedura in due fasi: prima calcoliamo le mediane per gruppo, poi le applichiamo ai missing.

In [5]:
# Calcolo delle mediane di Income per livello di Education
median_income_by_education = df.groupby('Education')['Income'].median().round(2)

print("Mediana del reddito per livello di istruzione:")
print(median_income_by_education.to_string())
print(f"\n(Per confronto — mediana globale: {df['Income'].median():,.2f} EUR)")

Mediana del reddito per livello di istruzione:
Education
2n Cycle      46805.0
Basic         20744.0
Graduation    51983.0
Master        50943.0
PhD           55185.0

(Per confronto — mediana globale: 51,371.00 EUR)


In [6]:
# Conteggio missing prima dell'imputazione
n_missing_before = df['Income'].isna().sum()

# Imputazione: per ogni riga con Income mancante, assegna la mediana del suo gruppo Education
df['Income'] = df.groupby('Education')['Income'].transform(
    lambda x: x.fillna(x.median())
)

# Verifica: dopo l'imputazione non deve esserci alcun NaN
n_missing_after = df['Income'].isna().sum()

print(f"Missing in Income prima dell'imputazione:  {n_missing_before}")
print(f"Missing in Income dopo l'imputazione:      {n_missing_after}")
print(f"\n{'OK - imputazione completata' if n_missing_after == 0 else 'ATTENZIONE: residuano missing'}")

Missing in Income prima dell'imputazione:  24
Missing in Income dopo l'imputazione:      0

OK - imputazione completata


### 📌 Nota metodologica: `transform` vs `fillna` diretto

Il pattern `groupby(...).transform(lambda x: x.fillna(x.median()))` è la forma idiomatica per imputazione condizionata in pandas. Rispetto a un approccio iterativo con loop esplicito:
- È **vettorializzato** (più veloce)
- Restituisce una Series allineata all'indice originale, quindi l'assegnazione è diretta
- Gestisce automaticamente i gruppi senza bisogno di costruire un dizionario manualmente

Questo dettaglio implementativo non cambia il risultato ma rende il codice più robusto e leggibile.

## 5. Conversione di `Dt_Customer` in datetime

La colonna `Dt_Customer` contiene date nel formato `DD-MM-YYYY` (es. `04-09-2012`), ma è attualmente memorizzata come stringa. Per poter calcolare durate, filtrare per periodo e ordinare cronologicamente, dobbiamo convertirla nel tipo `datetime`.

### Perché specifichiamo il formato esplicitamente

Il formato `04-09-2012` è ambiguo senza contesto: potrebbe essere il **4 settembre 2012** (formato europeo `DD-MM-YYYY`) o il **9 aprile 2012** (formato americano `MM-DD-YYYY`). Specificando `format='%d-%m-%Y'` eliminiamo ogni ambiguità.

In [7]:
# Conversione esplicita con formato DD-MM-YYYY
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')

# Verifica del range temporale coperto
print(f"Dtype della colonna: {df['Dt_Customer'].dtype}")
print(f"Data minima:  {df['Dt_Customer'].min().date()}")
print(f"Data massima: {df['Dt_Customer'].max().date()}")
print(f"Arco temporale coperto: {(df['Dt_Customer'].max() - df['Dt_Customer'].min()).days} giorni")

Dtype della colonna: datetime64[us]
Data minima:  2012-07-30
Data massima: 2014-06-29
Arco temporale coperto: 699 giorni


### 📌 Osservazione: finestra temporale

Il dataset copre registrazioni di clienti da metà 2012 a metà 2014, un arco di circa 2 anni. Questo conferma la scelta di usare **31 dicembre 2014** come data di riferimento per il calcolo di `Age` e `Customer_Tenure_Days`: è successiva all'ultima registrazione, quindi tutti i tenure risultano positivi, e la spesa osservata nei campi `Mnt*` (riferita agli ultimi 2 anni) copre plausibilmente fino a quella data.

## 6. Consolidamento di `Marital_Status`

La colonna `Marital_Status` contiene 8 categorie originali, alcune delle quali sono junk (`Absurd`, `YOLO`) o semanticamente ridondanti (`Alone` ≈ `Single`). Prima di consolidare, ispezioniamo le frequenze.

In [8]:
# Frequenza delle categorie originali
print("Distribuzione originale di Marital_Status:")
print(df['Marital_Status'].value_counts().to_string())

Distribuzione originale di Marital_Status:
Marital_Status
Married     864
Together    578
Single      479
Divorced    231
Widow        77
Alone         3
Absurd        2
YOLO          2


### Mappatura proposta

Consolidiamo in due categorie, allineate alla domanda di business (*un cliente condivide le decisioni di spesa con un partner?*):

| Categoria finale | Categorie originali |
|---|---|
| **`In_Couple`** | `Married`, `Together` |
| **`Single`** | `Single`, `Divorced`, `Widow`, `Alone`, `Absurd`, `YOLO` |

Questa dicotomia è ciò che conta davvero per il marketing: chi vive in coppia tende ad avere pattern di spesa (volumi, categorie preferite, sensibilità alle promozioni) diversi da chi vive solo, indipendentemente dal dettaglio legale dello stato civile.

In [9]:
# Dizionario di mappatura
marital_mapping = {
    'Married':  'In_Couple',
    'Together': 'In_Couple',
    'Single':   'Single',
    'Divorced': 'Single',
    'Widow':    'Single',
    'Alone':    'Single',
    'Absurd':   'Single',
    'YOLO':     'Single',
}

df['Marital_Status'] = df['Marital_Status'].map(marital_mapping)

# Verifica: nessun NaN residuo (cioè nessuna categoria imprevista)
assert df['Marital_Status'].isna().sum() == 0, "ATTENZIONE: categoria non mappata"

print("Distribuzione dopo il consolidamento:")
print(df['Marital_Status'].value_counts().to_string())
print(f"\nProporzione In_Couple: {(df['Marital_Status'] == 'In_Couple').mean():.1%}")

Distribuzione dopo il consolidamento:
Marital_Status
In_Couple    1442
Single        794

Proporzione In_Couple: 64.5%


## 7. Feature Engineering

Costruiamo ora le variabili derivate che renderanno più efficace l'analisi nel Notebook 03. Ogni feature ha una motivazione precisa legata all'obiettivo di business.

| Feature | Formula | Finalità analitica |
|---|---|---|
| `Age` | `2014 − Year_Birth` | Segmentazione per fascia d'età |
| `MntTotal` | Somma delle 6 colonne `Mnt*` | Proxy di Customer Lifetime Value |
| `TotalPurchases` | Somma delle 4 `Num*Purchases` | Volume totale di acquisti (escluse le visite web) |
| `TotalCampaignsAccepted` | Somma dei 5 `AcceptedCmp*` + `Response` | Misura di responsività alle campagne |
| `Customer_Tenure_Days` | `31-12-2014 − Dt_Customer` | Anzianità del cliente |
| `Family_Size` | `Kidhome + Teenhome + (1 o 2)` | Dimensione del nucleo familiare |
| `Has_Children` | `1 se Kidhome+Teenhome > 0 altrimenti 0` | Flag binario per analisi segmentate |

### Una precisazione importante su `TotalPurchases`

Sommiamo `NumDealsPurchases`, `NumWebPurchases`, `NumCatalogPurchases`, `NumStorePurchases`, ma **NON** `NumWebVisitsMonth`. Quest'ultima misura **visite** al sito (non acquisti) e su base mensile (non cumulativa): sommarla genererebbe una feature semanticamente incoerente.

In [10]:
# Data di riferimento (standard sul dataset: 31-12-2014)
REFERENCE_DATE = pd.Timestamp('2014-12-31')

# 1. Age
df['Age'] = REFERENCE_DATE.year - df['Year_Birth']

# 2. MntTotal — spesa totale su tutte le categorie di prodotto
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
            'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df['MntTotal'] = df[mnt_cols].sum(axis=1)

# 3. TotalPurchases — somma dei 4 canali di acquisto (NON include NumWebVisitsMonth)
purchase_cols = ['NumDealsPurchases', 'NumWebPurchases',
                 'NumCatalogPurchases', 'NumStorePurchases']
df['TotalPurchases'] = df[purchase_cols].sum(axis=1)

# 4. TotalCampaignsAccepted — 5 campagne storiche + Response all'ultima
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
                 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
df['TotalCampaignsAccepted'] = df[campaign_cols].sum(axis=1)

# 5. Customer_Tenure_Days — anzianita del cliente rispetto alla data di riferimento
df['Customer_Tenure_Days'] = (REFERENCE_DATE - df['Dt_Customer']).dt.days

# 6. Family_Size — figli + adulti (1 se single, 2 se in coppia)
df['Family_Size'] = (
    df['Kidhome']
    + df['Teenhome']
    + df['Marital_Status'].map({'Single': 1, 'In_Couple': 2})
)

# 7. Has_Children — flag binario
df['Has_Children'] = ((df['Kidhome'] + df['Teenhome']) > 0).astype(int)

print("Feature engineering completato.")
print(f"Dataset ora: {df.shape[0]:,} righe x {df.shape[1]} colonne (+7 feature create)")

Feature engineering completato.
Dataset ora: 2,236 righe x 34 colonne (+7 feature create)


### Verifica delle nuove feature

Controlliamo che i valori generati siano sensati: nessun valore negativo dove non dovrebbe esserci, distribuzioni plausibili, nessun NaN.

In [11]:
# Statistiche descrittive delle 7 nuove feature
new_features = ['Age', 'MntTotal', 'TotalPurchases', 'TotalCampaignsAccepted',
                'Customer_Tenure_Days', 'Family_Size', 'Has_Children']

df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,2236.0,45.101968,11.703281,18.0,37.00,44.0,55.0,74.0
MntTotal,2236.0,605.986583,601.865156,5.0,69.00,396.5,1045.5,2525.0
TotalPurchases,2236.0,14.872540,7.677874,0.0,8.00,15.0,21.0,44.0
TotalCampaignsAccepted,2236.0,0.447227,0.891113,0.0,0.00,0.0,1.0,5.0
Customer_Tenure_Days,2236.0,538.773256,202.181561,185.0,365.75,541.0,714.0,884.0
Family_Size,2236.0,2.595707,0.907468,1.0,2.00,3.0,3.0,5.0
Has_Children,2236.0,0.715116,0.451460,0.0,0.00,1.0,1.0,1.0


### 📌 Sanity check sulle feature create

Scorrendo i risultati sopra:

- **`Age`**: min 18, max 74 — range plausibile (dopo la rimozione degli outlier anagrafici)
- **`MntTotal`**: mediana ~400 EUR su 2 anni, max ~2.500 EUR — coerente con le componenti
- **`TotalPurchases`**: mediana ~12 acquisti su 2 anni — plausibile
- **`TotalCampaignsAccepted`**: max 5 su 6 possibili — nessun cliente ha accettato *tutte* le campagne, segnale di quanto sia difficile attivare lo stesso cliente ripetutamente
- **`Customer_Tenure_Days`**: min ~0, max ~900 giorni (~2.5 anni) — coerente con la finestra del dataset
- **`Family_Size`**: range 1-5 — plausibile
- **`Has_Children`**: media ~0.71 — circa il 71% dei clienti ha almeno un figlio

Tutto coerente. Nessun valore patologico.

## 8. Salvataggio del dataset pulito

Salviamo il DataFrame in `data/processed/` per renderlo disponibile al Notebook 03 senza dover ripetere tutte le operazioni di cleaning. Usiamo il formato CSV per mantenere coerenza con il raw (più portabile e ispezionabile) e lo salviamo in UTF-8.

In [12]:
import os

# Path di output — struttura parallela a data/raw/
OUTPUT_PATH = '../data/marketing_campaign_clean.csv'

# Creazione della directory se non esiste
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Salvataggio senza indice (l'indice di pandas non ha semantica qui)
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f"Dataset pulito salvato in:\n   {OUTPUT_PATH}")
print(f"\nDimensioni finali: {df.shape[0]:,} righe x {df.shape[1]} colonne")

Dataset pulito salvato in:
   ../data/marketing_campaign_clean.csv

Dimensioni finali: 2,236 righe x 34 colonne


## 9. Sintesi del Data Cleaning

### Trasformazioni applicate

| # | Operazione | Record impattati |
|---|---|---|
| 1 | Rimozione colonne costanti (`Z_CostContact`, `Z_Revenue`) | Tutte le righe (-2 colonne) |
| 2 | Rimozione outlier `Year_Birth` < 1940 | 3 righe rimosse |
| 3 | Rimozione outlier `Income` = 666.666 | 1 riga rimossa |
| 4 | Imputazione missing `Income` con mediana per `Education` | 24 valori imputati |
| 5 | Conversione `Dt_Customer` a `datetime` | Tutte le righe |
| 6 | Consolidamento `Marital_Status` in 2 categorie | Tutte le righe |
| 7 | Creazione di 7 feature derivate | +7 colonne |

### Confronto prima/dopo

- **Righe**: 2.240 → 2.236 (4 rimosse, 0.18% del totale)
- **Colonne**: 29 → 34 (−2 costanti, +7 derivate)
- **Missing values totali**: 24 → 0
- **Tipi di dato**: `Dt_Customer` ora è `datetime64`

### Decisioni documentate

Le scelte critiche di questo notebook — imputazione per gruppo, soglia outlier Year_Birth, rimozione chirurgica dell'outlier Income, consolidamento Marital_Status in dicotomia — sono state prese esplicitamente per **preservare il segnale analitico dei segmenti high-value** (clienti premium, partnership) che costituiscono il cuore delle raccomandazioni di business del progetto.

### Prossimo step — Notebook 03

Con il dataset pulito e arricchito, nel prossimo notebook procederemo a:
1. **EDA approfondita** sulle feature derivate (distribuzioni, correlazioni, visualizzazioni)
2. **Segmentazione clienti** — probabilmente con approccio RFM adattato (Recency, MntTotal, TotalPurchases) o clustering
3. **Analisi dell'efficacia delle campagne** per segmento
4. **Raccomandazioni finali** per l'allocazione del budget marketing

In [13]:
print("Notebook 02 — Data Cleaning & Feature Engineering completato.")
print(f"Dataset finale: {df.shape[0]:,} righe x {df.shape[1]} colonne")
print(f"Prossimo step: Notebook 03 — EDA, segmentazione e analisi campagne")

Notebook 02 — Data Cleaning & Feature Engineering completato.
Dataset finale: 2,236 righe x 34 colonne
Prossimo step: Notebook 03 — EDA, segmentazione e analisi campagne
